<a href="https://www.kaggle.com/code/shamanthakreddymallu/f1-dataset-creation?scriptVersionId=325837942" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports

In [ ]:
!pip install fastf1

In [ ]:
import fastf1
import pandas as pd
import os
import shutil
from datetime import datetime

# Configuration

In [ ]:
#schedule = fastf1.get_event_schedule(2017, include_testing=False)
#print(schedule[['RoundNumber', 'EventName', 'EventFormat', 'Session1', 'Session2', 'Session3', 'Session4', 'Session5']].to_string())

In [ ]:
YEAR      = 2022
ERA_DIR   = 'f1-ground-effect-hybrid-era'
CACHE_DIR = './fastf1_cache'

os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

# Helper Functions

In [ ]:
def td_to_sec(x):
    return round(x.total_seconds(), 3) if pd.notnull(x) else None


def fmt_race_time(x):
    if pd.isnull(x):
        return None
    total = x.total_seconds()
    h = int(total // 3600)
    m = int((total % 3600) // 60)
    s = total % 60
    return f"{h}:{m:02d}:{s:06.3f}" if h else f"{m}:{s:06.3f}"


def save_results(session, path):
    """Save session classification results."""
    cols_available = [c for c in [
        'DriverNumber', 'Abbreviation', 'FullName', 'TeamName',
        'GridPosition', 'Position', 'Points', 'Status', 'Time', 'Q1', 'Q2', 'Q3'
    ] if c in session.results.columns]

    results = session.results[cols_available].copy()
    rename = {
        'DriverNumber': 'driver_number', 'Abbreviation': 'abbreviation',
        'FullName': 'full_name', 'TeamName': 'team',
        'GridPosition': 'grid', 'Position': 'position',
        'Points': 'points', 'Status': 'status',
        'Time': 'time', 'Q1': 'q1', 'Q2': 'q2', 'Q3': 'q3'
    }
    results = results.rename(columns={k: v for k, v in rename.items() if k in results.columns})

    for col in ['time', 'q1', 'q2', 'q3']:
        if col in results.columns:
            results[col] = results[col].apply(fmt_race_time)

    results.to_csv(path, index=False)
    print(f"      results: {len(results)} drivers")


def save_laps(session, path):
    """Save lap times + telemetry for any session type."""
    lap_cols = [c for c in [
        'Driver', 'Team', 'LapNumber', 'LapTime',
        'Sector1Time', 'Sector2Time', 'Sector3Time',
        'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST',
        'Compound', 'TyreLife', 'FreshTyre', 'Stint',
        'PitInTime', 'PitOutTime', 'Position',
        'IsPersonalBest', 'TrackStatus'
    ] if c in session.laps.columns]

    laps = session.laps[lap_cols].copy()
    rename = {
        'Driver': 'driver', 'Team': 'team', 'LapNumber': 'lap',
        'LapTime': 'lap_time', 'Sector1Time': 's1', 'Sector2Time': 's2',
        'Sector3Time': 's3', 'SpeedI1': 'speed_i1', 'SpeedI2': 'speed_i2',
        'SpeedFL': 'speed_fl', 'SpeedST': 'speed_st', 'Compound': 'compound',
        'TyreLife': 'tyre_life', 'FreshTyre': 'fresh_tyre', 'Stint': 'stint',
        'PitInTime': 'pit_in', 'PitOutTime': 'pit_out', 'Position': 'position',
        'IsPersonalBest': 'is_personal_best', 'TrackStatus': 'track_status'
    }
    laps = laps.rename(columns={k: v for k, v in rename.items() if k in laps.columns})

    for col in ['lap_time', 's1', 's2', 's3', 'pit_in', 'pit_out']:
        if col in laps.columns:
            laps[col] = laps[col].apply(td_to_sec)

    # car telemetry aggregates per lap
    records = []
    for _, lap in session.laps.iterlaps():
        row = {'driver': lap['Driver'], 'lap': lap['LapNumber']}
        try:
            car = lap.get_car_data()
            row['full_throttle_pct'] = round(float((car['Throttle'] == 100).mean() * 100), 2)
            row['brake_pct']         = round(float((car['Brake'] > 0).mean() * 100), 2)
            row['gear_changes']      = int((car['nGear'].diff().fillna(0) != 0).sum())
            row['max_speed']         = round(float(car['Speed'].max()), 1)
            row['min_speed']         = round(float(car['Speed'].min()), 1)
            row['drs_activated']     = bool((car['DRS'] >= 10).any())
        except Exception:
            row.update({
                'full_throttle_pct': None, 'brake_pct': None,
                'gear_changes': None, 'max_speed': None,
                'min_speed': None, 'drs_activated': None
            })
        records.append(row)

    car_df = pd.DataFrame(records)
    laps = laps.merge(car_df, on=['driver', 'lap'], how='left')
    laps.to_csv(path, index=False)
    print(f"      laps: {len(laps)} laps, {len(laps.columns)} columns")


def collect_session(year, gp_name, session_key, laps_path, results_path):
    """Load a session and save both laps and results."""
    try:
        session = fastf1.get_session(year, gp_name, session_key)
        session.load(telemetry=True, weather=False, messages=False)
        save_results(session, results_path)
        save_laps(session, laps_path)
    except Exception as e:
        print(f"      ERROR: {e}")

# Main Code

In [ ]:
schedule = fastf1.get_event_schedule(YEAR, include_testing=False)
total = len(schedule)
print(f"\nF1 {YEAR} — {total} rounds found\n{'='*60}\n")

for _, event in schedule.iterrows():
    round_num = int(event['RoundNumber'])
    gp_name   = event['EventName']
    fmt       = event['EventFormat'].lower()
    is_sprint = 'sprint' in fmt
    gp_folder = f"{round_num:02d}_{gp_name.replace(' ', '_')}"
    OUT       = os.path.join(ERA_DIR, str(YEAR), gp_folder)
    os.makedirs(OUT, exist_ok=True)

    print(f"[{round_num:02d}/{total}] {gp_name} {'(Sprint)' if is_sprint else ''}")

    if event['EventDate'].date() > datetime.today().date():
        print("    upcoming race, skipping.\n")
        continue

    # sessions differ by weekend format
    if is_sprint:
        sessions = [
            ('FP1',  'practice_1_laps.csv',          'practice_1_results.csv'),
            ('SQ',   'sprint_qualifying_laps.csv',    'sprint_qualifying_results.csv'),
            ('S',    'sprint_laps.csv',               'sprint_results.csv'),
            ('Q',    'qualifying_laps.csv',           'qualifying_results.csv'),
            ('R',    'race_laps.csv',                 'race_results.csv'),
        ]
    else:
        sessions = [
            ('FP1',  'practice_1_laps.csv',   'practice_1_results.csv'),
            ('FP2',  'practice_2_laps.csv',   'practice_2_results.csv'),
            ('FP3',  'practice_3_laps.csv',   'practice_3_results.csv'),
            ('Q',    'qualifying_laps.csv',   'qualifying_results.csv'),
            ('R',    'race_laps.csv',          'race_results.csv'),
        ]

    for session_key, laps_file, results_file in sessions:
        print(f"    [{session_key}] {laps_file.replace('_laps.csv','').replace('_',' ')}")
        collect_session(
            YEAR, gp_name, session_key,
            os.path.join(OUT, laps_file),
            os.path.join(OUT, results_file)
        )

    print()

# Final Checks

In [ ]:
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
    print("Cache cleared.")

print(f"\nSeason download complete! Data saved under: {ERA_DIR}/{YEAR}/")